# Roads Thematic Accuracy

# !!! Indicator does not work


### Methods and Data
- Extrinsic approach.

#### Basis-DLM

The Basic DLM is published by the German Federal Agency for Cartography and Geodesy and describes the topographical
objects of the landscape and the relief of the Earth's surface in vector format. The objects are defined by their spatial
location, geometric type, descriptive attributes, and relationships to other objects. Each object has a unique 
identification number throughout Germany. The roads from this dataset are used for this indicator.

| Attribute | OSM tags  | DLM attribute | Description                                      | 
|-----------|-----------|---------------|--------------------------------------------------|
 | name      | name, ref | NAM           | The name or reference of a road.                 |
 | surface   | surface   | OFM           | The material the surface of the road is made of. |
 | lanes     | lanes     | FSZ           | The amount of lanes on this road segment.        |
 | width     | width     | BRF           | The width of the road.                           |
 | oneway    | oneway    | FAR           | The direction of oneway streets.                 |

### Processing

OSM roads and DLM roads are matched using [map-matching-2](https://github.com/addy90/map-matching-2) with a Markov decision process–based model.
For the matched roads first the presence of each attribute in both datasets is checked. If they are present in both,
the values are compared. By default, the values are compared directly, but there are some exceptions:

#### Name

For the name, the [Lhevenstein distance](https://en.wikipedia.org/wiki/Levenshtein_distance) was calculated. The names
were counted as a match when their Levensthein ratio was above 0.85.

#### Surface

For surface, OSM tags and DLM tags are matched. All OSM tags that are not in the following table, were counted as not matching.

| DLM value        | OSM tags                                                  |
|------------------|-----------------------------------------------------------|
| concrete         | concrete                                                  |
| bitumen, asphalt | asphalt                                                   |
| pavemed          | paving_stones, sett, brick, cobblestone, unhewn_cobblestone |
| rock, fragmented | fine_gravel, gravel, sand, compacted, pebblestone         |

#### Width

For the width, a tolerance of 1 m was applied.

#### Oneway

To check the direction of the road, the vector of both geometries is calculated. If both have the same sign it is counted as a match.


### References

- A. Wöltche, "Open source map matching with Markov decision processes: A new method and a detailed benchmark with existing approaches", Transactions in GIS, vol. 27, no. 7, pp. 1959–1991, Oct. 2023, doi: [10.1111/tgis.13107](https://onlinelibrary.wiley.com/doi/full/10.1111/tgis.13107).



In [1]:
import geojson
import requests
import geopandas as gpd
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys

indicator = "/roads-thematic-accuracy"
topic = "roads"
input_geom_path = "friedberg.geojson"
output_geom_path = "your_output_layer.gpkg"
max_workers = 20

base_url = "https://api.quality.ohsome.org/v1-test"
endpoint = "/indicators"
url = base_url + endpoint + indicator

gdf = gpd.read_file(input_geom_path)
gdf["result_value"] = pd.Series([None] * len(gdf), dtype="float")
gdf["response_time"] = pd.Series([None] * len(gdf), dtype="float")

headers = {"accept": "application/json"}

def fetch(index, geometry):
    bpolys = geojson.Feature(geometry=geometry)
    bpolys_collection = geojson.FeatureCollection([bpolys])

    parameters = {
        "topic": topic,
        "bpolys": bpolys_collection,
    }
    for attempt in range(5):
        try:
            print(f"posting request for index {index}")
            startresponse = time.time()
            response = requests.post(url, headers=headers, json=parameters, timeout=600)
            response.raise_for_status()
            result = response.json()
            endresponse = time.time()
            responsetime = endresponse - startresponse
            value = result["result"][0]["result"]["value"]
            return index, value, responsetime
        except requests.RequestException as e:
            print(f"Attempt {attempt + 1} failed at index {index}: {e}")
            if attempt < 3:
                print("Retrying...")
                time.sleep(2)
            else:
                print("Max retries reached. Skipping.")
                return index, None, None

start = time.time()
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(fetch, i, gdf.geometry.iloc[i]) for i in range(len(gdf))]

    for future in as_completed(futures):
        index, value, responsetime = future.result()
        gdf.at[index, "result_value"] = value
        gdf.at[index, "response_time"] = responsetime
        print(f"Completed index {index}: {value}")

end = time.time()
print(f"Calculation took {end - start:.2f} seconds")

gdf.to_file(output_geom_path, driver="GPKG")

posting request for index 0
Attempt 1 failed at index 0: HTTPSConnectionPool(host='api.quality.ohsome.org', port=443): Max retries exceeded with url: /v1-test/indicators/roads-thematic-accuracy (Caused by NameResolutionError("HTTPSConnection(host='api.quality.ohsome.org', port=443): Failed to resolve 'api.quality.ohsome.org' ([Errno -3] Temporary failure in name resolution)"))
Retrying...
posting request for index 0
Attempt 2 failed at index 0: HTTPSConnectionPool(host='api.quality.ohsome.org', port=443): Max retries exceeded with url: /v1-test/indicators/roads-thematic-accuracy (Caused by NameResolutionError("HTTPSConnection(host='api.quality.ohsome.org', port=443): Failed to resolve 'api.quality.ohsome.org' ([Errno -3] Temporary failure in name resolution)"))
Retrying...
posting request for index 0
Attempt 3 failed at index 0: HTTPSConnectionPool(host='api.quality.ohsome.org', port=443): Max retries exceeded with url: /v1-test/indicators/roads-thematic-accuracy (Caused by NameResolut